In [3]:
import polars as pl
from pathlib import Path

In [4]:
DATA_GENERAL = Path("../data_general")
DATA_PERSONAL = Path("../data_personal/Spotify Extended Streaming History")

In [5]:
general_data_frames = []
for num in range (10) :
    general_data_frames.append(pl.read_parquet(DATA_GENERAL / f'spotify_audio_features_{num}.parquet'))

In [6]:
display(general_data_frames[0].head())

id,name,popularity,null_response,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2Pe9cbhOTvOUTDE4bl7zzl""","""I dreamt you died""",0,0,630506,4,6,0,87.683,0.279,0.391,-12.054,0.32,0.816,0.737,0.177,0.0299
"""0wP732NKm8XgXu78XLRWoR""","""It's Death""",0,0,97216,4,5,1,105.298,0.429,0.318,-11.685,0.0566,0.587,0.782,0.202,0.36
"""22L6EJdnjx8oIo7GiF9hLe""","""Preliminary""",0,0,75180,4,0,1,117.657,0.283,0.581,-9.42,0.0555,0.923,0.939,0.106,0.0362
"""3a519lgQ13JXNi0G73mwMT""","""Disparage""",0,0,149447,4,5,0,100.685,0.244,0.995,-0.69,0.125,0.78,0.799,0.132,0.0634
"""27yP7p2lxWYTtnldRN8Kzx""","""Cut Down""",0,0,120816,4,7,1,123.499,0.313,0.618,0.411,0.073,0.843,0.109,0.126,0.187


In [7]:
personal_data_frames = {}
for num in range (2022,2027) :
    personal_data_frames[num] = pl.read_json(
        DATA_PERSONAL / f'Streaming_History_Audio_{num}.json',
        infer_schema_length=None  
        )

In [8]:
personal_data_frames[2022] =personal_data_frames[2022].with_columns(pl.col("spotify_track_uri").str.slice(14,22))

In [9]:
check_mus_id = personal_data_frames[2022]["spotify_track_uri"][0]
print(check_mus_id)

4u7EnebtmKWzUH433cf5Qv


In [10]:
#print(len(general_data_frames))
for g in range(10):
    general_data_frames[g] = general_data_frames[g].rename({ 'id' : 'spotify_track_uri' })
print(general_data_frames[0])

shape: (25_558_893, 17)
┌────────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬──────────┬─────────┐
│ spotify_tr ┆ name       ┆ popularit ┆ null_resp ┆ … ┆ acousticn ┆ instrumen ┆ liveness ┆ valence │
│ ack_uri    ┆ ---        ┆ y         ┆ onse      ┆   ┆ ess       ┆ talness   ┆ ---      ┆ ---     │
│ ---        ┆ str        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ f64      ┆ f64     │
│ str        ┆            ┆ i64       ┆ i64       ┆   ┆ f64       ┆ f64       ┆          ┆         │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪══════════╪═════════╡
│ 2Pe9cbhOTv ┆ I dreamt   ┆ 0         ┆ 0         ┆ … ┆ 0.816     ┆ 0.737     ┆ 0.177    ┆ 0.0299  │
│ OUTDE4bl7z ┆ you died   ┆           ┆           ┆   ┆           ┆           ┆          ┆         │
│ zl         ┆            ┆           ┆           ┆   ┆           ┆           ┆          ┆         │
│ 0wP732NKm8 ┆ It's Death ┆ 0         ┆ 0         ┆ … ┆ 0.587     ┆

In [11]:
status = False
mus_name = ''
for g in range(10):
    for url in general_data_frames[g]['spotify_track_uri']:
        if url == check_mus_id:
            status = True
            mus_name = general_data_frames[g].filter(pl.col('spotify_track_uri') == url).select(['name',])
            break
    if status :
        break

print(status, mus_name)

True shape: (1, 1)
┌─────────────────────────────────┐
│ name                            │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ Bohemian Rhapsody - Remastered… │
└─────────────────────────────────┘


# NOW LET`S FIND OUT TRACK

In [12]:
song_uri = '3U0UXxBIfjUsJ8RtxoxUFn'

In [ ]:

for g in range (10):
    if song_uri in general_data_frames[g]['spotify_track_uri']:
        #print(g)
        stats = general_data_frames[g].filter(pl.col('spotify_track_uri') == song_uri)
        break


2


In [19]:
display(stats)

spotify_track_uri,name,popularity,null_response,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""3U0UXxBIfjUsJ8RtxoxUFn""","""Monolith""",64,0,264656,4,11,0,142.013,0.508,0.952,-6.445,0.0405,0.000409,0.756,0.114,0.799


# I have priority on the  energy, instrumentalness and valence

In [46]:
# Writ None in coeficients if you want to not search for this parameters
# In cof`s writre cof like 

cof_tempo = None        #       diff in BPM
cof_dance = 0.3         #       from 0 to 1
cof_energy = 0.01       #       from 0 to 1
cof_loud = None         #       diff in dB (0 dB - max loudness)
cof_speech = 0.5       #       from 0 to 1
cof_acoustic = 0.5     #       from 0 to 1
cof_instr = 0.05        #       from 0 to 1 
cof_live = 0.2         #       from 0 to 1 
cof_valence = 0.07      #       from 0 to 1 


#energy = (float(stats['energy'][0]) - 0.02 , float(stats['energy'][0]) + 0.02 )
#instr = (float(stats['instrumentalness'][0]) - 0.02 , float(stats['instrumentalness'][0]) + 0.02 )
#valence = (float(stats['valence'][0]) - 0.02 , float(stats['valence'][0]) + 0.02 )

#print(energy)

In [41]:
result = general_data_frames[2]#

In [47]:
if cof_tempo:
    ...
if cof_dance:
    state = stats['danceability']
    result = result.filter((pl.col('danceability').is_between(state*(1 - cof_dance) ,state*(1 + cof_dance) )))
if cof_energy:
    state = stats['energy']
    result = result.filter((pl.col('energy').is_between(state*(1 - cof_energy) ,state*(1 + cof_energy) )))
if cof_loud:
    ...
if cof_speech:
    state = stats['speechiness']
    result = result.filter((pl.col('speechiness').is_between(state*(1 - cof_speech) ,state*(1 + cof_speech) )))
if cof_acoustic:
    state = stats['acousticness']
    result = result.filter((pl.col('acousticness').is_between(state*(1 - cof_acoustic) ,state*(1 + cof_acoustic) )))
if cof_instr:
    state = stats['instrumentalness']
    result = result.filter((pl.col('instrumentalness').is_between(state*(1 - cof_instr) ,state*(1 + cof_instr) )))
if cof_live:
    state = stats['liveness']
    result = result.filter((pl.col('liveness').is_between(state*(1 - cof_live) ,state*(1 + cof_live) )))
if cof_valence:
    state = stats['valence']
    result = result.filter((pl.col('valence').is_between(state*(1 - cof_valence) ,state*(1 + cof_valence) )))

In [ ]:
print(len(result))
display(result)

13


spotify_track_uri,name,popularity,null_response,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2y8kMVtqP10nSguUT384fC""","""Boom Boom (Instrumental)""",0,0,179108,4,11,1,129.991,0.547,0.953,-6.28,0.035,0.00029,0.753,0.103,0.84
"""20bsmr6r7ZCyFuSG00yQqX""","""Las cosas que hace - Version 8…",0,0,256003,4,11,1,119.998,0.53,0.951,-7.024,0.0381,0.000284,0.774,0.127,0.838
"""4qV8D077wGgggig1D9wHYC""","""200 Years Later""",0,0,290000,3,0,1,143.987,0.439,0.956,-4.361,0.0313,0.000476,0.778,0.116,0.781
"""24WFeTFFTkaK7ljDCMpTc5""","""Last Chance""",0,0,149600,4,4,1,130.001,0.638,0.944,-9.164,0.0518,0.000247,0.766,0.116,0.782
"""2RgF3rQn6OS2YF207zmC8X""","""Stay Tuned""",5,0,436966,4,9,0,145.01,0.634,0.952,-6.662,0.0523,0.000543,0.732,0.1,0.76
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""70B4CPzlNKa1rWwgmTcZOY""","""Tequila""",0,0,188639,4,6,0,129.973,0.652,0.957,-7.317,0.0435,0.000598,0.72,0.093,0.83
"""4T1oZ3UAFKbraSp5JAISWW""","""Monolith""",59,0,264656,4,11,0,142.013,0.508,0.952,-6.445,0.0405,0.000409,0.756,0.114,0.799
"""3U0UXxBIfjUsJ8RtxoxUFn""","""Monolith""",64,0,264656,4,11,0,142.013,0.508,0.952,-6.445,0.0405,0.000409,0.756,0.114,0.799
